In [1]:
import pandas as pd
import numpy as np
import re
import glob
import os
from typing import Tuple, List
import warnings
warnings.filterwarnings("ignore")

# =========================
# 0) PATHS (ALL SERVERS)
# =========================

PATH_META   = r"../../data/Riot_API/processed/match_detail/match_meta/*.parquet"
PATH_TEAMS  = r"../../data/Riot_API/processed/match_detail/teams/*.parquet"
PATH_PARTS  = r"../../data/Riot_API/processed/match_detail/participants/*.parquet"

PATH_EVENTS = r"../../data/Riot_API/processed/match_timeline/events/*.parquet"
PATH_FRAMES = r"../../data/Riot_API/processed/match_timeline/frames/*.parquet"

OUT_DM_TEAM  = r"../../data/Riot_API/feature/datamart/dm_match_team_all.csv"
OUT_DM_EARLY = r"../../data/Riot_API/feature/datamart/dm_match_team_early.csv"

# =========================
# 1) Baseline filters
# =========================
QUEUE_ALLOW = {420, 440}
MIN_DURATION_SEC = 900
END_RESULT_ALLOW = {"GameComplete"}
MIN_PATCH = (15, 1)
MAX_PATCH = (16, 2)

VALID_POS = ["TOP","JNG","MID","BOT","SUP"]
POS_MAP = {
    "TOP": "TOP",
    "JUNGLE": "JNG",
    "MIDDLE": "MID",
    "BOTTOM": "BOT",
    "UTILITY": "SUP",
}

AFK_COL = "ch_had_afk_teammate"
SURRENDER_COLS = ["game_ended_in_early_surrender","game_ended_in_surrender","team_early_surrendered"]

EARLY_MIN = 10

# =========================
# 2) Helpers
# =========================
def parse_major_minor(game_version: str):
    m = re.match(r"^\s*(\d+)\.(\d+)", str(game_version))
    if not m:
        return None
    return int(m.group(1)), int(m.group(2))

def patch_in_range(game_version: str):
    mm = parse_major_minor(game_version)
    if mm is None:
        return False
    return (mm >= MIN_PATCH) and (mm <= MAX_PATCH)

def safe_div(a, b):
    b = b.replace({0: np.nan})
    return (a / b).fillna(0.0)

def add_opponent_diff(df, key_cols, team_col, value_cols, suffix="_diff"):
    """
    Compute team vs opponent diff
    assumes team_id = 100 / 200
    """

    base = df[key_cols + [team_col] + value_cols].copy()

    # 상대팀 id 생성
    base["opp_team_id"] = base[team_col].map({100:200, 200:100})

    # opponent dataframe 생성
    opp = base.copy()
    opp = opp.rename(columns={team_col:"team_id_opp"})

    for c in value_cols:
        opp = opp.rename(columns={c:f"{c}_opp"})

    # merge
    merged = base.merge(
        opp[key_cols + ["team_id_opp"] + [f"{c}_opp" for c in value_cols]],
        left_on=key_cols + ["opp_team_id"],
        right_on=key_cols + ["team_id_opp"],
        how="left"
    )

    out = merged[key_cols + [team_col]].copy()

    for c in value_cols:
        out[f"{c}{suffix}"] = merged[c] - merged[f"{c}_opp"]

    return out

def team_pos_complete(parts: pd.DataFrame) -> pd.DataFrame:
    """
    Return match keys that satisfy:
      - both teams exist
      - each team has exactly 5 positions (TOP/JNG/MID/BOT/SUP) exactly once
      - team size = 5
    """
    # team size
    team_n = parts.groupby(["match_id","platform","team_id"]).size().reset_index(name="n_players")

    # position uniqueness
    pos_n = parts.groupby(["match_id","platform","team_id"])["pos5"].nunique().reset_index(name="n_pos")

    team_ok = team_n.merge(pos_n, on=["match_id","platform","team_id"], how="left")
    team_ok["team_ok"] = ((team_ok["n_players"]==5) & (team_ok["n_pos"]==5)).astype(int)

    # match ok if both teams ok
    match_ok = team_ok.groupby(["match_id","platform"])["team_ok"].min().reset_index(name="match_ok")
    return match_ok[match_ok["match_ok"]==1][["match_id","platform"]]

# =========================
# 3) Load CSV
# =========================

def load_parquet_glob(path_pattern):

    files = glob.glob(path_pattern)

    if len(files) == 0:
        raise ValueError(f"No files found: {path_pattern}")

    df_list = []

    for f in files:
        print("Loading:", os.path.basename(f))
        df = pd.read_parquet(f)
        df_list.append(df)

    df = pd.concat(df_list, ignore_index=True)

    return df


# =========================
# 3) Load parquet (glob)
# =========================

meta   = load_parquet_glob(PATH_META)
teams  = load_parquet_glob(PATH_TEAMS)
parts  = load_parquet_glob(PATH_PARTS)
events = load_parquet_glob(PATH_EVENTS)
frames = load_parquet_glob(PATH_FRAMES)

meta = meta.drop_duplicates(subset=["match_id","platform"])
teams = teams.drop_duplicates(subset=["match_id","platform","team_id"])
parts = parts.drop_duplicates(subset=["match_id","platform","participant_id"])

# =========================
# 4) Apply meta filters first
# =========================
meta = meta.drop_duplicates(subset=["match_id","platform"]).copy()
meta = meta[meta["queue_id"].isin(list(QUEUE_ALLOW))].copy()
meta["game_duration"] = pd.to_numeric(meta["game_duration"], errors="coerce")
meta = meta[meta["game_duration"] >= MIN_DURATION_SEC].copy()
meta = meta[meta["end_of_game_result"].isin(list(END_RESULT_ALLOW))].copy()
meta = meta[meta["game_version"].apply(patch_in_range)].copy()

keep_keys = meta[["match_id","platform"]].drop_duplicates()

teams  = teams.merge(keep_keys, on=["match_id","platform"], how="inner")
parts  = parts.merge(keep_keys, on=["match_id","platform"], how="inner")
events = events.merge(keep_keys, on=["match_id","platform"], how="inner")
frames = frames.merge(keep_keys, on=["match_id","platform"], how="inner")

# =========================
# 5) Quality filters: surrender/AFK/pos complete (match-level)
# =========================
# normalize positions
parts["pos5"] = parts["team_position"].astype(str).str.upper().map(POS_MAP)
parts = parts[parts["pos5"].isin(VALID_POS)].copy()

# drop AFK matches if col exists
if AFK_COL in parts.columns:
    parts[AFK_COL] = pd.to_numeric(parts[AFK_COL], errors="coerce").fillna(0).astype(int)
    bad_afk = parts.groupby(["match_id","platform"])[AFK_COL].max().reset_index()
    bad_afk = bad_afk[bad_afk[AFK_COL] > 0][["match_id","platform"]]
    if len(bad_afk) > 0:
        keep_keys = keep_keys.merge(bad_afk.assign(_bad=1), on=["match_id","platform"], how="left")
        keep_keys = keep_keys[keep_keys["_bad"].isna()][["match_id","platform"]]

# drop surrender matches if cols exist in participants
for c in SURRENDER_COLS:
    if c in parts.columns:
        parts[c] = pd.to_numeric(parts[c], errors="coerce").fillna(0).astype(int)
        bad_s = parts.groupby(["match_id","platform"])[c].max().reset_index()
        bad_s = bad_s[bad_s[c] > 0][["match_id","platform"]]
        if len(bad_s) > 0:
            keep_keys = keep_keys.merge(bad_s.assign(_bad=1), on=["match_id","platform"], how="left")
            keep_keys = keep_keys[keep_keys["_bad"].isna()][["match_id","platform"]]

# enforce both teams 5pos complete
keep_pos = team_pos_complete(parts)
keep_keys = keep_keys.merge(keep_pos, on=["match_id","platform"], how="inner")

# re-filter all tables
meta   = meta.merge(keep_keys, on=["match_id","platform"], how="inner")
teams  = teams.merge(keep_keys, on=["match_id","platform"], how="inner")
parts  = parts.merge(keep_keys, on=["match_id","platform"], how="inner")
events = events.merge(keep_keys, on=["match_id","platform"], how="inner")
frames = frames.merge(keep_keys, on=["match_id","platform"], how="inner")

# =========================
# 6) dm_match_team (Full / Model C)
# =========================

# Identify numeric participant columns to aggregate (aggressive: almost all numeric except IDs/strings/meta duplicates)
ID_LIKE = {
    "match_id","platform","participant_id","puuid","team_id",
    "team_position","individual_position","role","champion_id","champion_name","pos5"
}
# meta duplicates in participants we don't want as features (we'll rely on meta/teams)
META_DUPE = {"queue_id","game_duration","game_version","game_mode","game_creation","win","time_played"}

cand_cols = [c for c in parts.columns if c not in ID_LIKE and c not in META_DUPE]
# keep only numeric-like
num_cols = []
for c in cand_cols:
    if parts[c].dtype.kind in "biufc":  # already numeric
        num_cols.append(c)
    else:
        # try coercing; if many numeric after coercion -> accept
        tmp = pd.to_numeric(parts[c], errors="coerce")
        if tmp.notna().mean() > 0.7:
            parts[c] = tmp
            num_cols.append(c)

# team totals
team_tot = parts.groupby(["match_id","platform","team_id"], as_index=False)[num_cols].sum(min_count=1)

# team win target from teams (cleaner than participant win)
# teams has 'win' already as team-level
target = teams[["match_id","platform","team_id","win"]].rename(columns={"win":"win_team"})
team_tot = team_tot.merge(target, on=["match_id","platform","team_id"], how="left")

# position pivot for all numeric cols
pos_piv = (
    parts.groupby(["match_id","platform","team_id","pos5"], as_index=False)[num_cols]
    .sum(min_count=1)
)

# pivot to wide: col_POS
wide_list = []
for c in num_cols:
    w = pos_piv.pivot_table(index=["match_id","platform","team_id"], columns="pos5", values=c, aggfunc="sum", fill_value=0).reset_index()
    w.columns = ["match_id","platform","team_id"] + [f"{c}_{p}" for p in VALID_POS]
    wide_list.append(w)

pos_wide = wide_list[0]
for w in wide_list[1:]:
    pos_wide = pos_wide.merge(w, on=["match_id","platform","team_id"], how="left")

# shares for each numeric col (team total basis)
share_df = pos_wide[["match_id","platform","team_id"]].copy()
for c in num_cols:
    if c not in team_tot.columns:
        continue
    for p in VALID_POS:
        share_df[f"{c}_share_{p}"] = safe_div(pos_wide[f"{c}_{p}"], team_tot[c])

# risk metrics across positions for each numeric col (full aggressive)
risk_df = pos_wide[["match_id","platform","team_id"]].copy()
for c in num_cols:
    cols = [f"{c}_{p}" for p in VALID_POS]
    if not all(col in pos_wide.columns for col in cols):
        continue
    mat = pos_wide[cols].astype(float)
    mean = mat.mean(axis=1)
    std = mat.std(axis=1, ddof=0)
    med = mat.median(axis=1)
    q90 = mat.quantile(0.90, axis=1)
    q10 = mat.quantile(0.10, axis=1)
    std_safe = std.replace({0: np.nan})
    z = (mat.sub(mean, axis=0)).div(std_safe, axis=0)

    risk_df[f"{c}_team_mean"] = mean
    risk_df[f"{c}_team_std"] = std
    risk_df[f"{c}_team_cv"] = safe_div(std, mean.abs())
    risk_df[f"{c}_spread_ratio"] = safe_div((q90-q10), med.replace({0: np.nan})).fillna(0.0)
    risk_df[f"{c}_extreme_rate"] = (z.abs() >= 2.0).sum(axis=1) / 5.0

# diff for team totals (num_cols)
diff_df = add_opponent_diff(team_tot, key_cols=["match_id","platform"], team_col="team_id", value_cols=num_cols, suffix="_diff")

# carry_dependency / concentration based on damage share + gold share if present
struct_df = team_tot[["match_id","platform","team_id"]].copy()
dmg_base = "total_damage_dealt_to_champions"
gold_base = "gold_earned"

if all(f"{dmg_base}_share_{p}" in share_df.columns for p in VALID_POS):
    shares = share_df[[f"{dmg_base}_share_{p}" for p in VALID_POS]].astype(float)
    struct_df["carry_dependency_damage"] = shares.max(axis=1)
    struct_df["damage_concentration_index"] = (shares**2).sum(axis=1)

if all(f"{gold_base}_share_{p}" in share_df.columns for p in VALID_POS):
    shares = share_df[[f"{gold_base}_share_{p}" for p in VALID_POS]].astype(float)
    struct_df["carry_dependency_gold"] = shares.max(axis=1)
    struct_df["gold_concentration_index"] = (shares**2).sum(axis=1)

# objectives from teams (already clean + includes horde=voidgrub)
obj_cols = [c for c in teams.columns if c.startswith("obj_")]
teams_obj = teams[["match_id","platform","team_id"] + obj_cols].copy()

# objective_total (kills only)
kill_cols = [c for c in obj_cols if c.endswith("_kills")]
teams_obj["objective_total"] = teams_obj[kill_cols].sum(axis=1, skipna=True)

# objective_diff
obj_diff = add_opponent_diff(
    teams_obj[["match_id","platform","team_id","objective_total"]],
    key_cols=["match_id","platform"], team_col="team_id", value_cols=["objective_total"], suffix="_diff"
).rename(columns={"objective_total_diff":"objective_diff"})

# 결측도 0 처리
teams_obj["obj_atakhan_kills"] = teams_obj["obj_atakhan_kills"].fillna(0)

# diff 생성
atakhan_diff = add_opponent_diff(
    teams_obj[["match_id","platform","team_id","obj_atakhan_kills"]],
    key_cols=["match_id","platform"],
    team_col="team_id",
    value_cols=["obj_atakhan_kills"],
    suffix="_diff"
).rename(columns={"obj_atakhan_kills_diff":"atakhan_kills_diff"})

# merge meta for filters
meta_small = meta[["match_id","platform","queue_id","game_duration","game_mode","game_version","game_creation"]].copy()

# assemble dm_match_team
dm_team = (
    teams[["match_id","platform","team_id","win"]]  # keep team win too
    .rename(columns={"win":"win_team"})
    .merge(meta_small, on=["match_id","platform"], how="left")
    .merge(team_tot, on=["match_id","platform","team_id"], how="left", suffixes=("","_drop"))
    .merge(pos_wide, on=["match_id","platform","team_id"], how="left")
    .merge(share_df, on=["match_id","platform","team_id"], how="left")
    .merge(diff_df, on=["match_id","platform","team_id"], how="left")
    .merge(risk_df, on=["match_id","platform","team_id"], how="left")
    .merge(struct_df, on=["match_id","platform","team_id"], how="left")
    .merge(teams_obj, on=["match_id","platform","team_id"], how="left")
    .merge(obj_diff, on=["match_id","platform","team_id"], how="left")
    .merge(atakhan_diff, on=["match_id","platform","team_id"], how="left")
)

# drop any accidental *_drop duplicates
drop_cols = [c for c in dm_team.columns if c.endswith("_drop")]
if drop_cols:
    dm_team = dm_team.drop(columns=drop_cols)

na = dm_team.isna().mean().sort_values(ascending=False)

# 100% NaN 컬럼 드랍
drop_all_nan = na[na >= 0.999].index.tolist()

dm_team = dm_team.drop(columns=drop_all_nan)

ch_cols = [c for c in dm_team.columns if c.startswith("ch_")]
dm_team[ch_cols] = dm_team[ch_cols].fillna(0)

dm_team.to_csv(OUT_DM_TEAM, index=False, encoding="utf-8-sig")
print("Saved:", OUT_DM_TEAM, dm_team.shape)

# =========================
# 7) dm_match_team_early (Early / Model A)
# =========================

# -------------------------------------------------
# (A) events team_id 재구성 (killer / creator 기반)
# -------------------------------------------------

# 0은 player가 아님 → NaN 처리
for c in ["killer_id","creator_id"]:
    if c in events.columns:
        events[c] = pd.to_numeric(events[c], errors="coerce")
        events.loc[events[c] == 0, c] = np.nan

pmap = parts[["match_id","platform","participant_id","team_id"]].drop_duplicates()

# killer 기반 매핑
events = events.merge(
    pmap,
    left_on=["match_id","platform","killer_id"],
    right_on=["match_id","platform","participant_id"],
    how="left",
    suffixes=("","_killer")
)

events = events.rename(columns={"team_id_killer":"team_id_killer"})

# creator 기반 매핑
events = events.merge(
    pmap,
    left_on=["match_id","platform","creator_id"],
    right_on=["match_id","platform","participant_id"],
    how="left",
    suffixes=("","_creator")
)

events = events.rename(columns={"team_id_creator":"team_id_creator"})

# 최종 team_id 생성
events["team_id"] = events["team_id_killer"].fillna(events["team_id_creator"])

events.drop(
    ["participant_id_killer","participant_id_creator",
     "team_id_killer","team_id_creator"],
    axis=1,
    inplace=True,
    errors="ignore"
)

# -------------------------------------------------
# (B) frames → team early stats
# -------------------------------------------------

# frames: add team_id via participants mapping
frames2 = frames.merge(pmap, on=["match_id","platform","participant_id"], how="left")

# frames@10 team sums
f10 = frames2[frames2["minute"] == EARLY_MIN].copy()

f10["cs_10_part"] = (
    pd.to_numeric(f10["minions_killed"], errors="coerce").fillna(0) +
    pd.to_numeric(f10["jungle_minions_killed"], errors="coerce").fillna(0)
)

early_team = (
    f10.groupby(["match_id","platform","team_id"], as_index=False)
    .agg(
        gold_10=("total_gold","sum"),
        xp_10=("xp","sum"),
        cs_10=("cs_10_part","sum"),
        dmg_to_champ_10=("dmg_total_done_to_champions","sum"),
        dmg_taken_10=("dmg_total_taken","sum"),
        level_10=("level","sum"),
        enemy_cc_time_10=("time_enemy_spent_controlled","sum"),
    )
)

# milliseconds → seconds
early_team["enemy_cc_time_10"] = early_team["enemy_cc_time_10"] / 1000.0

# -------------------------------------------------
# (C) early frame diff
# -------------------------------------------------

early_frame_diff = add_opponent_diff(
    early_team,
    key_cols=["match_id","platform"],
    team_col="team_id",
    value_cols=[
        "gold_10","xp_10","cs_10",
        "dmg_to_champ_10","dmg_taken_10",
        "level_10","enemy_cc_time_10"
    ],
    suffix="_diff"
)

# -------------------------------------------------
# (D) Early 이벤트 집계
# -------------------------------------------------

events["minute"] = pd.to_numeric(events["minute"], errors="coerce")
e10 = events[events["minute"] <= EARLY_MIN].copy()

# 주요 event types만 사용
EARLY_EVENT_TYPES = [
    "CHAMPION_KILL",
    "WARD_PLACED",
    "WARD_KILL",
    "ELITE_MONSTER_KILL",
    "TURRET_PLATE_DESTROYED"
]

e10 = e10[e10["event_type"].isin(EARLY_EVENT_TYPES)]

# -------------------------------------------------
# (D-1) First Blood
# -------------------------------------------------

fb = e10[e10["event_type"] == "CHAMPION_KILL"].copy()

# timestamp 기준 가장 빠른 kill
fb = (
    fb.sort_values(["match_id","platform","timestamp"])
      .groupby(["match_id","platform"], as_index=False)
      .first()
)

# first blood team
fb = fb[["match_id","platform","team_id"]].copy()
fb["first_blood"] = 1

# 팀 기준으로 정렬
teams_key = teams[["match_id","platform","team_id"]].drop_duplicates()

fb = teams_key.merge(fb, on=["match_id","platform","team_id"], how="left")
fb["first_blood"] = fb["first_blood"].fillna(0).astype(int)

# diff 계산
fb_diff = add_opponent_diff(
    fb,
    key_cols=["match_id","platform"],
    team_col="team_id",
    value_cols=["first_blood"],
    suffix="_diff"
)

# event count
type_counts = (
    e10.groupby(["match_id","platform","team_id","event_type"])
    .size()
    .unstack(fill_value=0)
    .reset_index()
)

# rename
for c in list(type_counts.columns):
    if c not in ["match_id","platform","team_id"]:
        type_counts = type_counts.rename(columns={c: f"evt_{str(c).lower()}_10"})

evt_cols = [c for c in type_counts.columns if c.startswith("evt_")]

evt_diff = add_opponent_diff(
    type_counts,
    key_cols=["match_id","platform"],
    team_col="team_id",
    value_cols=evt_cols,
    suffix="_diff"
)

# -------------------------------------------------
# (E) Elite monster (dragon / baron / herald / horde)
# -------------------------------------------------

elite = e10[e10["event_type"] == "ELITE_MONSTER_KILL"].copy()

if "monster_type" in elite.columns:

    elite["monster_type"] = elite["monster_type"].str.lower()

    elite_piv = (
        elite.groupby(["match_id","platform","team_id","monster_type"])
        .size()
        .unstack(fill_value=0)
        .reset_index()
    )

    for c in list(elite_piv.columns):
        if c not in ["match_id","platform","team_id"]:
            elite_piv = elite_piv.rename(columns={c: f"elite_{str(c).lower()}_10"})

    elite_cols = [c for c in elite_piv.columns if c.startswith("elite_")]

    elite_diff = add_opponent_diff(
        elite_piv,
        key_cols=["match_id","platform"],
        team_col="team_id",
        value_cols=elite_cols,
        suffix="_diff"
    )

else:
    elite_diff = None

# -------------------------------------------------
# (F) assemble early dm
# -------------------------------------------------

dm_early = (
    early_team[["match_id","platform","team_id"]]
    .merge(meta_small, on=["match_id","platform"], how="left")
    .merge(target, on=["match_id","platform","team_id"], how="left")
    .merge(early_frame_diff, on=["match_id","platform","team_id"], how="left")
    .merge(evt_diff, on=["match_id","platform","team_id"], how="left")
    .merge(fb_diff, on=["match_id","platform","team_id"], how="left")
)
dm_early = dm_early.sort_values(["match_id","platform","team_id"])

if elite_diff is not None:
    dm_early = dm_early.merge(
        elite_diff,
        on=["match_id","platform","team_id"],
        how="left"
    )

elite_cols = [c for c in dm_early.columns if c.startswith("elite_") and c.endswith("_diff_10")]
dm_early[elite_cols] = dm_early[elite_cols].fillna(0)
dm_early = dm_early.drop(columns=["evt_elite_monster_kill_10_diff"])

dm_early.to_csv(OUT_DM_EARLY, index=False, encoding="utf-8-sig")
print("Saved:", OUT_DM_EARLY, dm_early.shape)

Loading: details_br1_part1_meta.parquet
Loading: details_eun1_part1_meta.parquet
Loading: details_euw1_part1_meta.parquet
Loading: details_euw1_part2_meta.parquet
Loading: details_jp1_part1_meta.parquet
Loading: details_kr_part1_meta.parquet
Loading: details_la1_part1_meta.parquet
Loading: details_la2_part1_meta.parquet
Loading: details_me1_part1_meta.parquet
Loading: details_na1_part1_meta.parquet
Loading: details_oc1_part1_meta.parquet
Loading: details_ru_part1_meta.parquet
Loading: details_sg2_part1_meta.parquet
Loading: details_tr1_part1_meta.parquet
Loading: details_tw2_part1_meta.parquet
Loading: details_vn2_part1_meta.parquet
Loading: details_br1_part1_teams.parquet
Loading: details_eun1_part1_teams.parquet
Loading: details_euw1_part1_teams.parquet
Loading: details_euw1_part2_teams.parquet
Loading: details_jp1_part1_teams.parquet
Loading: details_kr_part1_teams.parquet
Loading: details_la1_part1_teams.parquet
Loading: details_la2_part1_teams.parquet
Loading: details_me1_part1_te